In [ ]:
%pip install ultralytics opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import math
import os
import time 

from ultralytics import YOLO
from IPython.display  import display,Video,Image
import matplotlib.pyplot as plt

print("all the libraies import successfully")

In [ ]:
VIDEO_PATH="Data/input/fiteness2.mp4"

OUTPUT_PATH="Data/output"
os.makedirs("output",exist_ok=True)

#chech 
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"video not found:{VIDEO_PATH}")

In [ ]:
MODEL_PATH = "yolov8n-pose.pt"

model = YOLO(MODEL_PATH)

print("YOLOv8-Pose model loaded successfully!")

In [ ]:
LEFT_BODY_CONNECTIONS=[(5,7),(7,9),(5,11),(11,13),(13,15)]
RIGHT_BODY_CONNECTIONS=[(6,8),(8,10),(6,12),(12,14),(14,16)]
MID_BODY_CONNECTIONS=[(5,6),(11,12),(5,11),(6,12)]

# Define skeleton colors

LEFT_COLOR = (0, 255, 0)
RIGHT_COLOR = (0, 0, 255)
MID_COLOR = (255, 0, 0)

KEYPOINT_COLOR = (0, 255, 255)

print("All skeleton colors defined successfully!")

In [ ]:
def estimate_body_orientation(keypoints, confidence, threshold=0.25):

    """
    Estimate approximate body orientation.

    Returns:
        roll  - Sideways shoulder tilt
        pitch - Forward/backward body tilt estimate
        yaw   - Left/right facing estimate
    """

    def valid_point(index):
        return (
            confidence[index] >= threshold
            and np.all(np.isfinite(keypoints[index]))
            and not np.all(keypoints[index] == 0)
        )

    # Required body points
    required = [0, 5, 6, 11, 12]

    if not all(valid_point(i) for i in required):
        return 0.0, 0.0, 0.0

    nose = keypoints[0]

    left_shoulder = keypoints[5]
    right_shoulder = keypoints[6]

    left_hip = keypoints[11]
    right_hip = keypoints[12]

    # Calculate shoulder midpoint
    shoulder_mid = (left_shoulder + right_shoulder) / 2

    # Calculate hip midpoint
    hip_mid = (left_hip + right_hip) / 2

    # Shoulder width
    shoulder_width = np.linalg.norm(
        right_shoulder - left_shoulder
    )

    if shoulder_width < 1:
        return 0.0, 0.0, 0.0

    # ROLL: Angle of shoulder line
    dx = right_shoulder[0] - left_shoulder[0]
    dy = right_shoulder[1] - left_shoulder[1]

    roll = math.degrees(math.atan2(dy, dx))

    # PITCH: Estimate from spine direction
    spine_vector = hip_mid - shoulder_mid

    pitch = math.degrees(
        math.atan2(spine_vector[0], spine_vector[1])
    )

    # YAW: Approximate from nose horizontal offset
    nose_offset = nose[0] - shoulder_mid[0]

    yaw = math.degrees(
        math.atan2(nose_offset, shoulder_width / 2)
    )

    # Limit extreme values
    roll = np.clip(roll, -45, 45)
    pitch = np.clip(pitch, -45, 45)
    yaw = np.clip(yaw, -65, 65)

    return float(roll), float(pitch), float(yaw)


print("Body orientation function created!")

In [ ]:
def create_rotation_matrix(roll, pitch, yaw):

    # Convert degrees to radians
    rx = math.radians(roll)
    ry = math.radians(pitch)
    rz = math.radians(yaw)

    # Rotation around X-axis
    Rx = np.array([
        [1, 0, 0],
        [0, math.cos(rx), -math.sin(rx)],
        [0, math.sin(rx), math.cos(rx)]
    ])

    # Rotation around Y-axis
    Ry = np.array([
        [math.cos(ry), 0, math.sin(ry)],
        [0, 1, 0],
        [-math.sin(ry), 0, math.cos(ry)]
    ])

    # Rotation around Z-axis
    Rz = np.array([
        [math.cos(rz), -math.sin(rz), 0],
        [math.sin(rz), math.cos(rz), 0],
        [0, 0, 1]
    ])

    # Combine rotations
    R = Rz @ Ry @ Rx

    return R


print("3D rotation matrix function created!")

In [ ]:
def draw_3d_features(
    frame,
    bbox,
    keypoints,
    confidence,
    track_id=None
):

    x1, y1, x2, y2 = map(int, bbox)

    box_w = max(20, x2 - x1)
    box_h = max(20, y2 - y1)

    # Estimate body orientation
    roll, pitch, yaw = estimate_body_orientation(
        keypoints, confidence
    )

    # Center of the cuboid
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    # Estimated cuboid dimensions
    box_width = box_w * 0.75
    box_height = box_h * 0.90
    box_depth = box_w * 0.35

    # Half dimensions
    w = box_width / 2
    h = box_height / 2
    d = box_depth / 2

    # Eight cuboid vertices
    vertices = np.array([
        [-w, -h, -d],
        [ w, -h, -d],
        [ w,  h, -d],
        [-w,  h, -d],

        [-w, -h,  d],
        [ w, -h,  d],
        [ w,  h,  d],
        [-w,  h,  d]
    ], dtype=np.float32)

    # Rotate vertices
    R = create_rotation_matrix(roll, pitch, yaw)

    rotated = vertices @ R.T

    # Fake perspective projection
    # This is only a visual approximation, not real camera projection.
    projected = []

    for point in rotated:

        px, py, pz = point

        screen_x = center_x + px + pz * 0.35
        screen_y = center_y + py - pz * 0.15

        projected.append([
            int(screen_x),
            int(screen_y)
        ])

    projected = np.array(projected, dtype=np.int32)

    # Cuboid edges
    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7)
    ]

    # Draw cuboid
    for start, end in edges:

        cv2.line(
            frame,
            tuple(projected[start]),
            tuple(projected[end]),
            (255, 255, 0),
            2,
            cv2.LINE_AA
        )

    # Draw XYZ axis triad
    axis_length = max(25, int(box_w * 0.20))

    origin = np.array([center_x, center_y, 0])

    axes = [
        (np.array([axis_length, 0, 0]), (0, 0, 255), "X"),
        (np.array([0, -axis_length, 0]), (0, 255, 0), "Y"),
        (np.array([0, 0, axis_length]), (255, 0, 0), "Z")
    ]

    def project_point(point):

        px, py, pz = point

        return (
            int(center_x + px + pz * 0.35),
            int(center_y + py - pz * 0.15)
        )

    origin_2d = project_point([0, 0, 0])

    for axis, color, label in axes:

        rotated_axis = R @ axis

        endpoint = project_point(rotated_axis)

        cv2.arrowedLine(
            frame,
            origin_2d,
            endpoint,
            color,
            2,
            tipLength=0.25
        )

        cv2.putText(
            frame,
            label,
            endpoint,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

    # Display Roll, Pitch, Yaw
    text_x = max(5, x1)
    text_y = max(25, y1 - 35)

    cv2.putText(
        frame,
        f"R: {roll:.1f} P: {pitch:.1f} Y: {yaw:.1f}",
        (text_x, text_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 255),
        2
    )

    return frame


print("3D cuboid drawing function created!")

In [ ]:
def draw_pose(frame, keypoints, confidence, threshold=0.25):

    # Draw body connections
    connections = [
        (LEFT_BODY_CONNECTIONS, LEFT_COLOR),
        (RIGHT_BODY_CONNECTIONS, RIGHT_COLOR),
        (MID_BODY_CONNECTIONS, MID_COLOR)
    ]

    for connection_list, color in connections:

        for start, end in connection_list:

            if (
                confidence[start] >= threshold
                and confidence[end] >= threshold
            ):

                p1 = tuple(keypoints[start].astype(int))
                p2 = tuple(keypoints[end].astype(int))

                cv2.line(
                    frame,
                    p1,
                    p2,
                    color,
                    2,
                    cv2.LINE_AA
                )

    # Draw individual keypoints
    for i, point in enumerate(keypoints):

        if confidence[i] < threshold:
            continue

        x, y = map(int, point)

        cv2.circle(
            frame,
            (x, y),
            4,
            KEYPOINT_COLOR,
            -1
        )

    return frame


print("Pose drawing function created!")

In [ ]:
def process_frame(frame, results):

    for result in results:

        if result.boxes is None:
            continue

        if result.keypoints is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()

        keypoints_all = result.keypoints.xy.cpu().numpy()

        # Keypoint confidence
        if result.keypoints.conf is not None:
            confidence_all = result.keypoints.conf.cpu().numpy()
        else:
            confidence_all = np.ones(
                (len(keypoints_all), 17),
                dtype=np.float32
            )

        # Tracking IDs
        if result.boxes.id is not None:
            track_ids = result.boxes.id.cpu().numpy().astype(int)
        else:
            track_ids = [None] * len(boxes)

        for bbox, keypoints, confidence, track_id in zip(
            boxes,
            keypoints_all,
            confidence_all,
            track_ids
        ):

            # Draw skeleton
            frame = draw_pose(
                frame,
                keypoints,
                confidence
            )

            # Draw 3D cuboid
            frame = draw_3d_features(
                frame,
                bbox,
                keypoints,
                confidence,
                track_id
            )

            # Draw bounding box
            x1, y1, x2, y2 = map(int, bbox)

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )

            # Display tracking ID
            label = (
                f"Person ID: {track_id}"
                if track_id is not None
                else "Person ID: pending"
            )

            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

    return frame


print("Frame processing function created!")

In [ ]:
# Open input video
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Unable to open input video!")

# Video properties
fps = cap.get(cv2.CAP_PROP_FPS)

if not np.isfinite(fps) or fps <= 0:
    fps = 25.0

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video Information")
print("-------------------------")
print("Width:", frame_width)
print("Height:", frame_height)
print("FPS:", fps)
print("Total frames:", total_frames)

# Ensure output directory exists
os.makedirs("outputs", exist_ok=True)

# Try MP4 first
OUTPUT_PATH = os.path.abspath(
    "outputs/yolo_pose_3d_output.mp4"
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (frame_width, frame_height)
)

# If MP4 fails, try AVI
if not out.isOpened():

    out.release()

    print("MP4 writer failed. Trying AVI format...")

    OUTPUT_PATH = os.path.abspath(
        "outputs/yolo_pose_3d_output.avi"
    )

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")

    out = cv2.VideoWriter(
        OUTPUT_PATH,
        fourcc,
        fps,
        (frame_width, frame_height)
    )

if not out.isOpened():

    cap.release()
    out.release()

    raise RuntimeError(
        "Unable to create MP4 or AVI video. "
        "Check folder permissions and OpenCV installation."
    )

print("Output writer successfully initialized!")
print("Saving to:", OUTPUT_PATH)

frame_id = 0
start_time = time.time()

print("\nStarting video processing...\n")

try:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_id += 1

        # YOLO detection and tracking
        results = model.track(
            source=frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=0.25,
            iou=0.7,
            verbose=False
        )

        # Draw skeleton and 3D cuboid
        frame_out = process_frame(
            frame.copy(),
            results
        )

        # Calculate FPS
        elapsed = time.time() - start_time

        current_fps = frame_id / max(elapsed, 0.001)

        # Display FPS
        cv2.putText(
            frame_out,
            f"FPS: {current_fps:.2f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

        # Write output
        out.write(frame_out)

        if frame_id % 50 == 0:

            print(
                f"Processed {frame_id}/{total_frames} frames"
            )

finally:

    cap.release()
    out.release()

print("\nProcessing completed!")
print("Total frames:", frame_id)
print("Output saved at:", OUTPUT_PATH)
print("File exists:", os.path.exists(OUTPUT_PATH))

In [ ]:
if not os.path.exists(OUTPUT_PATH):
    raise FileNotFoundError("Output video was not created!")

print("Displaying processed video...")

display(
    Video(
        OUTPUT_PATH,
        embed=True,
        width=800
    )
)

In [ ]:
cap = cv2.VideoCapture(OUTPUT_PATH)

ret, frame = cap.read()

cap.release()

if ret:

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    plt.figure(figsize=(14, 8))

    plt.imshow(frame_rgb)

    plt.title("YOLOv8-Pose + 3D Human Visualization")

    plt.axis("off")

    plt.show()

else:

    print("Unable to read output frame.")